# Pothole Segmentation (YOLOv8): 9-model evaluation, graphs and Excel report
**Runtime:** Runtime > Change runtime type > **T4 GPU**. Run cells top to bottom.

**How this is organised (follows your master prompt)**
1. Cells 1-3: setup + dataset inspection/validation. **No training happens here.**
2. Cell 4: `APPROVE_TRAINING = False`. Experiments start only after you set it to `True`.
3. Then: patch dataset -> features -> boosting models (4 x 7 feature sets = 28 combinations) -> MobileNetV2 / ResNet50 -> YOLO (3 models) -> graphs -> Excel.

**Methodology (three different task types, reported separately)**
- **YOLOv8n-seg, YOLOv8s-seg, YOLO11n-seg:** trained and evaluated as detection + instance segmentation (Ultralytics box and mask P/R/mAP).
- **MobileNetV2, ResNet50:** ImageNet-pretrained; (a) fine-tuned end-to-end as classifiers, (b) frozen as feature extractors for the boosting models.
- **AdaBoost, GradientBoosting, XGBoost, LightGBM:** trained on extracted image features (HOG, HSV colour histogram, LBP, GLCM, all handcrafted combined, MobileNetV2 and ResNet50 embeddings).
- **Dataset note:** all training images contain potholes (no negative images), so classifiers are trained on *patches*: pothole crops vs. pothole-free road crops cut from the same images. Accuracy/F1 of classifiers and P/R/mAP of YOLO are therefore **not directly comparable**.
- Nothing is fabricated: missing/not-applicable metrics stay blank (never 0).

## 1. Setup

In [ ]:
!pip -q install ultralytics xgboost lightgbm scikit-image openpyxl seaborn

In [ ]:
# ===== SETUP: config, upload, unzip (original zip is never modified), folder structure =====
import os, sys, json, zipfile, shutil, time, random, platform, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

BASE = Path("/content/pothole_project")
ZIP_PATH = "/content/Pothole_Segmentation_YOLOv8.zip"
# Option: load from Google Drive instead of uploading
# from google.colab import drive; drive.mount('/content/drive')
# ZIP_PATH = "/content/drive/MyDrive/Pothole_Segmentation_YOLOv8.zip"

if not os.path.exists(ZIP_PATH):
    from google.colab import files
    up = files.upload()                      # choose Pothole_Segmentation_YOLOv8.zip
    ZIP_PATH = "/content/" + list(up.keys())[0]

for d in ["src/evaluation", "src/feature_extraction", "src/visualization", "src/utils", "scripts",
          "results/graphs/model_comparison", "results/graphs/yolo_segmentation", "results/graphs/classical_ml",
          "results/graphs/deep_learning", "results/graphs/confusion_matrices", "results/metrics",
          "results/final_report", "models/classical", "models/deep_learning", "logs", "features"]:
    (BASE / d).mkdir(parents=True, exist_ok=True)

DS_DIR = BASE / "dataset"
if not DS_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DS_DIR)
DS = next(p.parent for p in DS_DIR.rglob("data.yaml"))   # folder that contains data.yaml
print("Dataset root:", DS)
print("Top-level items:", sorted(os.listdir(DS)))

## 2. Phase 1-6: inspection and dataset validation (no training)

In [ ]:
# ===== PHASE 1-2: inspection + dataset validation (no training) =====
import yaml, numpy as np, pandas as pd, importlib
from PIL import Image

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
cfg = yaml.safe_load(open(DS / "data.yaml"))
names = cfg["names"]; names = list(names.values()) if isinstance(names, dict) else list(names)
nc = cfg["nc"]

def poly_area(pts):
    x, y = pts[:, 0], pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def validate_split(split):
    imgs = sorted(p for p in (DS / split / "images").iterdir() if p.suffix.lower() in IMG_EXT)
    lbls = sorted((DS / split / "labels").glob("*.txt"))
    istem, lstem = {p.stem for p in imgs}, {p.stem for p in lbls}
    issues, n_inst, bad_img = [], 0, 0
    class_counts = {}
    sizes = {}
    for p in imgs:
        try:
            with Image.open(p) as im:
                sizes[im.size] = sizes.get(im.size, 0) + 1
        except Exception:
            bad_img += 1
    for lp in lbls:
        lines = [l.split() for l in open(lp).read().strip().splitlines() if l.strip()]
        if not lines: issues.append(f"{lp.name}: empty label file")
        for t in lines:
            c = int(float(t[0])); vals = np.array(t[1:], dtype=float)
            class_counts[c] = class_counts.get(c, 0) + 1; n_inst += 1
            if c < 0 or c >= nc: issues.append(f"{lp.name}: class id {c} out of range")
            if len(vals) < 6 or len(vals) % 2: issues.append(f"{lp.name}: malformed polygon ({len(vals)} values)")
            elif vals.min() < 0 or vals.max() > 1: issues.append(f"{lp.name}: coordinates outside [0,1]")
            elif poly_area(vals.reshape(-1, 2)) <= 0: issues.append(f"{lp.name}: zero-area polygon")
    return dict(images=len(imgs), labels=len(lbls), instances=n_inst,
                images_without_label=sorted(istem - lstem)[:10], labels_without_image=sorted(lstem - istem)[:10],
                unreadable_images=bad_img, class_counts=class_counts, image_sizes={str(k): v for k, v in sizes.items()},
                n_issues=len(issues), issue_examples=issues[:10])

report = {"classes": names, "nc": nc, "train": validate_split("train"), "valid": validate_split("valid")}
report["passed"] = all(report[s]["n_issues"] == 0 and report[s]["unreadable_images"] == 0 and
                       not report[s]["images_without_label"] and not report[s]["labels_without_image"]
                       for s in ["train", "valid"])

# environment
import torch
ver = {"python": platform.python_version()}
for m in ["numpy", "pandas", "sklearn", "xgboost", "lightgbm", "torch", "torchvision", "ultralytics", "skimage", "cv2"]:
    try: ver[m] = importlib.import_module(m).__version__
    except Exception: ver[m] = "not installed"
ver["cuda_available"] = torch.cuda.is_available()
ver["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU) - Runtime > Change runtime type > GPU"
report["environment"] = ver
json.dump(report, open(BASE / "results/metrics/dataset_validation.json", "w"), indent=2, default=str)

print("=== DATASET ===")
print("Classes:", names)
for s in ["train", "valid"]:
    r = report[s]
    print(f"{s:5s}: {r['images']} images | {r['labels']} labels | {r['instances']} polygon instances | issues: {r['n_issues']} | sizes: {r['image_sizes']}")
print("Validation passed:", report["passed"])
print("\n=== ENVIRONMENT ===")
for k, v in ver.items(): print(f"{k:16s}{v}")
print("\n=== EXISTING EVALUATION RESULTS ===")
print("found" if (BASE / "results/evaluation_results.csv").exists() else "none yet (no experiments have been run)")
print("Pending experiments: all 9 models")
print("\nNOTE: every training image contains at least one pothole, so there are no negative images. "
      "Classification models therefore use PATCH-level classification (pothole patch vs. pothole-free road patch), built below.")

## STOP: review the report above
Set `APPROVE_TRAINING = True` in the next cell when you approve the experiments.

In [ ]:
# ===== APPROVAL GATE: nothing below runs until you explicitly approve =====
APPROVE_TRAINING = False        # <-- change to True ONLY when you approve training / experiments

RUN_CLASSICAL = True            # 4 boosting models x 7 feature sets = 28 combinations
RUN_DEEP      = True            # fine-tuned MobileNetV2 and ResNet50
RUN_YOLO      = True            # YOLOv8n-seg, YOLOv8s-seg, YOLO11n-seg
YOLO_EPOCHS   = 50
DL_EPOCHS     = 10
SEED          = 42

assert APPROVE_TRAINING, ("Training is not approved yet. Phases 1-6 (inspection/validation) are done. "
                          "Set APPROVE_TRAINING = True in this cell and run again to start the experiments.")

import numpy as np, pandas as pd, cv2, joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
random.seed(SEED); np.random.seed(SEED)
RES = BASE / "results"

MODEL_ORDER = [("YOLOv8n-seg", "YOLO Segmentation"), ("YOLOv8s-seg", "YOLO Segmentation"), ("YOLO11n-seg", "YOLO Segmentation"),
               ("MobileNetV2", "Deep Learning"), ("ResNet50", "Deep Learning"),
               ("AdaBoost", "Classical ML"), ("GradientBoosting", "Classical ML"), ("XGBoost", "Classical ML"), ("LightGBM", "Classical ML")]
SUMMARY_FEATURE_SET = "Handcrafted_All"   # fixed in advance for the 9-model summary (not picked from validation results)
COLS = ["model", "category", "feature_set", "accuracy", "precision", "recall", "f1", "roc_auc", "map50", "map50_95",
        "inference_ms", "feature_extraction_ms", "model_size_mb", "eval_status", "metric_basis"]

def upsert(path, rows, keys):
    """Add/replace rows in a CSV keyed by `keys`; keeps results of earlier runs. Missing values stay NaN (never 0)."""
    new = pd.DataFrame(rows)
    if Path(path).exists():
        old = pd.read_csv(path)
        mk = lambda d: d[keys].astype(str).agg("|".join, axis=1)
        old = old[~mk(old).isin(set(mk(new)))]
        new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)
    return new

def refresh_summary():
    """Rebuild results/evaluation_results.csv/.json: 9 rows; models without results get status 'not_evaluated'."""
    combos = pd.read_csv(RES / "all_combinations.csv") if (RES / "all_combinations.csv").exists() else pd.DataFrame(columns=COLS)
    rows = []
    for m, cat in MODEL_ORDER:
        if cat == "Classical ML":
            hit = combos[(combos.model == m) & (combos.feature_set == SUMMARY_FEATURE_SET)]
        else:
            hit = combos[(combos.model == m) & (combos.feature_set == "end-to-end")] if cat == "Deep Learning" else \
                  combos[(combos.model == m)]
        if len(hit): rows.append(hit.iloc[0].to_dict())
        else: rows.append({"model": m, "category": cat, "eval_status": "not_evaluated"})
    df = pd.DataFrame(rows).reindex(columns=COLS)
    df.to_csv(RES / "evaluation_results.csv", index=False)
    df.to_json(RES / "evaluation_results.json", orient="records", indent=2)   # NaN -> null
    return df

def clf_metrics(y, pred, prob):
    return dict(accuracy=accuracy_score(y, pred), precision=precision_score(y, pred, zero_division=0),
                recall=recall_score(y, pred, zero_division=0), f1=f1_score(y, pred, zero_division=0),
                roc_auc=roc_auc_score(y, prob))
print("Approved. Experiments enabled.")

## 3. Classification patch dataset

In [ ]:
# ===== CLASSIFICATION DATASET: pothole patches (positive) vs pothole-free road patches (negative) =====
# Positives: crop around each polygon's bounding box (+15% margin).  Negatives: random patches of the SAME size as a
# positive box in the same image, accepted only if <1% of the patch overlaps ANY pothole mask. Train/valid kept separate.
PATCH, MIN_SIDE, MARGIN = 224, 24, 0.15

def build_patches(split):
    rng = np.random.default_rng(SEED)
    X, y, meta = [], [], []
    skipped = 0
    for ip in sorted((DS / split / "images").iterdir()):
        if ip.suffix.lower() not in IMG_EXT: continue
        lp = DS / split / "labels" / (ip.stem + ".txt")
        img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
        polys = []
        for line in open(lp):
            t = line.split()
            if len(t) >= 7: polys.append(np.array(t[1:], dtype=float).reshape(-1, 2) * [W, H])
        full = np.zeros((H, W), np.uint8)
        for p in polys: cv2.fillPoly(full, [p.astype(np.int32)], 1)
        boxes = []
        for p in polys:
            x0, y0 = p.min(0); x1, y1 = p.max(0); w, h = x1 - x0, y1 - y0
            if w < MIN_SIDE or h < MIN_SIDE: skipped += 1; continue
            bx = [int(max(0, x0 - w * MARGIN)), int(max(0, y0 - h * MARGIN)), int(min(W, x1 + w * MARGIN)), int(min(H, y1 + h * MARGIN))]
            boxes.append(bx)
            X.append(cv2.resize(img[bx[1]:bx[3], bx[0]:bx[2]], (PATCH, PATCH))); y.append(1); meta.append((ip.name, 1, *bx))
        for bx in boxes:
            bw, bh = bx[2] - bx[0], bx[3] - bx[1]
            for _ in range(50):
                nx, ny = int(rng.integers(0, W - bw + 1)), int(rng.integers(0, H - bh + 1))
                if full[ny:ny + bh, nx:nx + bw].mean() < 0.01:
                    X.append(cv2.resize(img[ny:ny + bh, nx:nx + bw], (PATCH, PATCH))); y.append(0); meta.append((ip.name, 0, nx, ny, nx + bw, ny + bh)); break
    print(f"{split}: {len(X)} patches | pothole={int(np.sum(y))} background={len(y)-int(np.sum(y))} | skipped tiny polygons={skipped}")
    return np.stack(X).astype(np.uint8), np.array(y), pd.DataFrame(meta, columns=["image", "label", "x0", "y0", "x1", "y1"])

Xtr, ytr, mtr = build_patches("train")
Xva, yva, mva = build_patches("valid")
np.savez_compressed(BASE / "features/patches.npz", Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva)
mtr.to_csv(BASE / "features/patches_train_meta.csv", index=False); mva.to_csv(BASE / "features/patches_valid_meta.csv", index=False)

## 4. Feature extraction

In [ ]:
# ===== FEATURE EXTRACTION: handcrafted (HOG, colour, LBP, GLCM) + CNN embeddings (MobileNetV2, ResNet50) =====
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
import torch, torchvision

def _gray(rgb, s): return cv2.cvtColor(cv2.resize(rgb, (s, s)), cv2.COLOR_RGB2GRAY)
def f_hog(rgb):   return hog(_gray(rgb, 128), orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2))
def f_color(rgb):
    hsv = cv2.cvtColor(cv2.resize(rgb, (128, 128)), cv2.COLOR_RGB2HSV)
    out = []
    for c, hi in enumerate([180, 256, 256]):
        h = cv2.calcHist([hsv], [c], None, [16], [0, hi]).ravel(); out.append(h / (h.sum() + 1e-9))
    return np.concatenate(out)
def f_lbp(rgb):
    l = local_binary_pattern(_gray(rgb, 128), 8, 1, "uniform")
    h, _ = np.histogram(l, bins=10, range=(0, 10)); return h / (h.sum() + 1e-9)
def f_glcm(rgb):
    q = (_gray(rgb, 64) // 8).astype(np.uint8)             # 32 grey levels
    g = graycomatrix(q, [1, 3], [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4], levels=32, symmetric=True, normed=True)
    return np.concatenate([graycoprops(g, p).ravel() for p in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation"]])

HAND = {"HOG": f_hog, "ColorHist": f_color, "LBP": f_lbp, "GLCM": f_glcm}
dev = "cuda" if torch.cuda.is_available() else "cpu"
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
def to_tensor(xb):   # uint8 NHWC numpy -> normalised NCHW float tensor on device
    t = torch.from_numpy(xb).to(dev).permute(0, 3, 1, 2).float() / 255.0
    return (t - MEAN) / STD

def build_backbone(name):
    if name == "MobileNetV2":
        m = torchvision.models.mobilenet_v2(weights="IMAGENET1K_V1"); m.classifier = torch.nn.Identity()
    else:
        m = torchvision.models.resnet50(weights="IMAGENET1K_V2"); m.fc = torch.nn.Identity()
    return m.to(dev)

@torch.no_grad()
def embed(model, X, bs=64):
    model.eval(); out = []
    for i in range(0, len(X), bs): out.append(model(to_tensor(X[i:i + bs])).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def ms_per_image(model, X, n=150):
    """Latency for batch size 1 (after warm-up), in ms/image."""
    model.eval(); xs = [to_tensor(X[i:i + 1]) for i in range(min(n, len(X)))]
    for x in xs[:10]: model(x)
    if dev == "cuda": torch.cuda.synchronize()
    t = time.perf_counter()
    for x in xs: model(x)
    if dev == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t) / len(xs) * 1000

FEATS, FEAT_MS, GROUP_SLICES = {}, {}, {}
for name, fn in HAND.items():
    t = time.perf_counter(); ftr = np.stack([fn(x) for x in Xtr]); t_tr = time.perf_counter() - t
    t = time.perf_counter(); fva = np.stack([fn(x) for x in Xva]); t_va = time.perf_counter() - t
    FEATS[name] = (ftr, fva); FEAT_MS[name] = t_va / len(Xva) * 1000
    print(f"{name:10s} dims={ftr.shape[1]:5d}  {FEAT_MS[name]:.2f} ms/img")

ftr = np.hstack([FEATS[k][0] for k in HAND]); fva = np.hstack([FEATS[k][1] for k in HAND])
FEATS["Handcrafted_All"] = (ftr, fva); FEAT_MS["Handcrafted_All"] = sum(FEAT_MS[k] for k in HAND)
o = 0
for k in HAND: d = FEATS[k][0].shape[1]; GROUP_SLICES[k] = (o, o + d); o += d

for name in ["MobileNetV2", "ResNet50"]:
    bb = build_backbone(name)
    etr, eva = embed(bb, Xtr), embed(bb, Xva)
    FEATS[name + "_emb"] = (etr, eva); FEAT_MS[name + "_emb"] = ms_per_image(bb, Xva)
    print(f"{name+'_emb':16s} dims={etr.shape[1]:5d}  {FEAT_MS[name+'_emb']:.2f} ms/img ({dev}, batch=1)")
    del bb; torch.cuda.empty_cache() if dev == "cuda" else None
FEATURE_SETS = list(FEATS)
print("Feature sets:", FEATURE_SETS)

## 5. Classical ML: all model x feature-set combinations

In [ ]:
# ===== CLASSICAL ML: 4 boosting models x all feature sets (all combinations) =====
if RUN_CLASSICAL:
    from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
    from xgboost import XGBClassifier
    from lightgbm import LGBMClassifier

    def make_models():
        return {"AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=SEED),
                "GradientBoosting": GradientBoostingClassifier(n_estimators=200, random_state=SEED),
                "XGBoost": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                                         eval_metric="logloss", tree_method="hist", random_state=SEED, n_jobs=-1),
                "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.1, num_leaves=31, subsample=0.8, subsample_freq=1,
                                           colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)}
    PRED = {"y_true": yva}
    rows, imp_rows, hist = [], [], []
    for fs in FEATURE_SETS:
        ftr_, fva_ = FEATS[fs]
        for mname, clf in make_models().items():
            t0 = time.perf_counter(); clf.fit(ftr_, ytr); fit_s = time.perf_counter() - t0
            t = time.perf_counter(); prob = clf.predict_proba(fva_)[:, 1]; pred_ms = (time.perf_counter() - t) / len(fva_) * 1000
            pred = (prob >= 0.5).astype(int)
            p = BASE / f"models/classical/{mname}__{fs}.joblib"; joblib.dump(clf, p)
            r = dict(model=mname, category="Classical ML", feature_set=fs, **clf_metrics(yva, pred, prob),
                     map50=np.nan, map50_95=np.nan, inference_ms=FEAT_MS[fs] + pred_ms, feature_extraction_ms=FEAT_MS[fs],
                     model_size_mb=p.stat().st_size / 1e6, eval_status="completed",
                     metric_basis="patch-level binary classification (validation split), threshold 0.5; inference = feature extraction + predict")
            rows.append(r); PRED[f"prob__{mname}__{fs}"] = prob
            print(f"{mname:17s}{fs:16s} acc={r['accuracy']:.3f} f1={r['f1']:.3f} auc={r['roc_auc']:.3f} fit={fit_s:.0f}s")
            if fs == "Handcrafted_All":
                imp = np.asarray(clf.feature_importances_, float); imp = imp / (imp.sum() + 1e-12)
                for g, (a, b) in GROUP_SLICES.items():
                    imp_rows.append(dict(model=mname, feature_set=fs, group=g, importance=float(imp[a:b].sum())))
    upsert(RES / "all_combinations.csv", rows, ["model", "feature_set"])
    upsert(RES / "metrics/feature_importance.csv", imp_rows, ["model", "feature_set", "group"])
    old = dict(np.load(RES / "metrics/predictions.npz")) if (RES / "metrics/predictions.npz").exists() else {}
    old.update(PRED); np.savez_compressed(RES / "metrics/predictions.npz", **old)
    refresh_summary(); print("Classical ML done.")

## 6. MobileNetV2 and ResNet50 (fine-tuned)

In [ ]:
# ===== DEEP LEARNING: fine-tune MobileNetV2 and ResNet50 (end-to-end) =====
if RUN_DEEP:
    import torch.nn as nn
    def finetune(name, epochs=DL_EPOCHS, bs=32, lr=1e-4):
        torch.manual_seed(SEED)
        m = build_backbone(name)
        if name == "MobileNetV2": m.classifier = nn.Linear(1280, 2)
        else: m.fc = nn.Linear(2048, 2)
        m = m.to(dev); opt = torch.optim.Adam(m.parameters(), lr=lr); ce = nn.CrossEntropyLoss()
        rng = np.random.default_rng(SEED); hist = []
        def run_eval():
            m.eval(); losses, probs = [], []
            with torch.no_grad():
                for i in range(0, len(Xva), 64):
                    lo = m(to_tensor(Xva[i:i + 64])); yb = torch.from_numpy(yva[i:i + 64]).to(dev)
                    losses.append(ce(lo, yb).item() * len(yb)); probs.append(torch.softmax(lo, 1)[:, 1].cpu().numpy())
            return sum(losses) / len(Xva), np.concatenate(probs)
        for ep in range(epochs):
            m.train(); idx = rng.permutation(len(Xtr)); tl = 0; corr = 0
            for i in range(0, len(idx), bs):
                b = idx[i:i + bs]; xb = to_tensor(Xtr[b]); yb = torch.from_numpy(ytr[b]).to(dev)
                flip = torch.rand(len(b), device=dev) < 0.5; xb[flip] = xb[flip].flip(-1)   # horizontal-flip augmentation
                opt.zero_grad(); lo = m(xb); loss = ce(lo, yb); loss.backward(); opt.step()
                tl += loss.item() * len(b); corr += (lo.argmax(1) == yb).sum().item()
            vl, vp = run_eval()
            hist.append(dict(model=name, epoch=ep + 1, train_loss=tl / len(Xtr), val_loss=vl, train_acc=corr / len(Xtr),
                             val_acc=accuracy_score(yva, (vp >= .5).astype(int))))
            print(f"{name} ep{ep+1:02d} train_loss={hist[-1]['train_loss']:.4f} val_loss={vl:.4f} val_acc={hist[-1]['val_acc']:.3f}")
        _, prob = run_eval()           # final-epoch weights are used (no checkpoint picked on the validation set)
        p = BASE / f"models/deep_learning/{name}_finetuned.pt"; torch.save(m.state_dict(), p)
        lat = ms_per_image(m, Xva)
        r = dict(model=name, category="Deep Learning", feature_set="end-to-end", **clf_metrics(yva, (prob >= .5).astype(int), prob),
                 map50=np.nan, map50_95=np.nan, inference_ms=lat, feature_extraction_ms=np.nan, model_size_mb=p.stat().st_size / 1e6,
                 eval_status="completed", metric_basis=f"patch-level binary classification (validation split), fine-tuned {epochs} epochs; latency batch=1 on {dev}")
        return r, prob, hist
    rows, PRED, H = [], {}, []
    for n in ["MobileNetV2", "ResNet50"]:
        r, prob, h = finetune(n); rows.append(r); PRED[f"prob__{n}__end-to-end"] = prob; H += h
    PRED["y_true"] = yva
    upsert(RES / "all_combinations.csv", rows, ["model", "feature_set"])
    upsert(RES / "metrics/dl_history.csv", H, ["model", "epoch"])
    old = dict(np.load(RES / "metrics/predictions.npz")) if (RES / "metrics/predictions.npz").exists() else {}
    old.update(PRED); np.savez_compressed(RES / "metrics/predictions.npz", **old)
    refresh_summary(); print("Deep learning done.")

## 7. YOLO segmentation (longest step: roughly 15-45 min per model on a T4)

In [ ]:
# ===== YOLO SEGMENTATION: train + evaluate with Ultralytics (object detection AND instance segmentation metrics) =====
if RUN_YOLO:
    from ultralytics import YOLO
    # new yaml with absolute paths; the original data.yaml is left untouched
    YAML = BASE / "dataset_yolo.yaml"
    YAML.write_text(f"path: {DS}\ntrain: train/images\nval: valid/images\nnames:\n  0: Pothole\n")
    YOLO_MODELS = {"YOLOv8n-seg": "yolov8n-seg.pt", "YOLOv8s-seg": "yolov8s-seg.pt", "YOLO11n-seg": "yolo11n-seg.pt"}
    RUNS = RES / "yolo_runs"
    f1 = lambda p, r: 2 * p * r / (p + r) if (p + r) > 0 else np.nan
    rows, detail = [], []
    for key, weights in YOLO_MODELS.items():
        YOLO(weights).train(data=str(YAML), epochs=YOLO_EPOCHS, imgsz=640, batch=16, seed=SEED, project=str(RUNS), name=key,
                            exist_ok=True, plots=True, verbose=False)
        best = RUNS / key / "weights/best.pt"
        m = YOLO(str(best)).val(data=str(YAML), split="val", imgsz=640, batch=1, plots=True, project=str(RUNS), name=key + "_val", exist_ok=True)
        sp = m.speed; ms = sp["preprocess"] + sp["inference"] + sp["postprocess"]
        b, s = m.box, m.seg
        detail.append(dict(model=key, box_precision=b.mp, box_recall=b.mr, box_f1=f1(b.mp, b.mr), box_map50=b.map50, box_map50_95=b.map,
                           mask_precision=s.mp, mask_recall=s.mr, mask_f1=f1(s.mp, s.mr), mask_map50=s.map50, mask_map50_95=s.map,
                           preprocess_ms=sp["preprocess"], inference_ms=sp["inference"], postprocess_ms=sp["postprocess"]))
        rows.append(dict(model=key, category="YOLO Segmentation", feature_set="end-to-end", accuracy=np.nan,
                         precision=s.mp, recall=s.mr, f1=f1(s.mp, s.mr), roc_auc=np.nan, map50=s.map50, map50_95=s.map,
                         inference_ms=ms, feature_extraction_ms=np.nan, model_size_mb=best.stat().st_size / 1e6, eval_status="completed",
                         metric_basis="instance MASK metrics from Ultralytics val (validation split; P/R at max-F1 confidence; mAP at IoU 0.5 and 0.5:0.95); box metrics in yolo_detail"))
        print(key, "mask mAP50=%.3f mAP50-95=%.3f | box mAP50=%.3f" % (s.map50, s.map, b.map50))
    upsert(RES / "all_combinations.csv", rows, ["model", "feature_set"])
    upsert(RES / "yolo_detail.csv", detail, ["model"])
    refresh_summary(); print("YOLO done.")

## 8. Graphs (300 DPI) from the saved results

In [ ]:
%%writefile /content/pothole_project/scripts/generate_graphs.py
"""Reusable graph generator. Reads ONLY saved evaluation files; never invents values.
Usage: python scripts/generate_graphs.py [--base /content/pothole_project]
Re-run any time new results are added; unavailable metrics/models are skipped and reported."""
import argparse, json, shutil, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc

ap = argparse.ArgumentParser(); ap.add_argument("--base", default="/content/pothole_project"); a = ap.parse_args()
BASE = Path(a.base); RES = BASE / "results"; G = RES / "graphs"
sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
plt.rcParams.update({"savefig.dpi": 300, "figure.dpi": 100, "axes.titleweight": "bold", "axes.titlesize": 13})
CAT_COL = {"YOLO Segmentation": "#d1495b", "Deep Learning": "#2e86ab", "Classical ML": "#66a182"}
NOTE = ("YOLO precision/recall/mAP = instance-mask metrics (IoU-matched, Ultralytics). Accuracy/precision/recall/F1/AUC of classifiers = "
        "patch-level pothole vs background. Different tasks: not strictly comparable.")
done, skipped = [], []

def save(fig, sub, name):
    d = G / sub; d.mkdir(parents=True, exist_ok=True); p = d / name
    fig.savefig(p, dpi=300, bbox_inches="tight"); plt.close(fig); done.append(str(p.relative_to(BASE)))

def skip(what, why): skipped.append(f"{what}: {why}")

def load(name, required=False):
    p = RES / name
    if p.exists(): return pd.read_csv(p)
    if required: sys.exit(f"Missing required file {p}")
    return None

# ---------- load + validate ----------
summ = load("evaluation_results.csv")
if summ is None and (RES / "evaluation_results.json").exists(): summ = pd.read_json(RES / "evaluation_results.json")
if summ is None: sys.exit("No evaluation_results.csv/json found - run the experiments first.")
need = {"model", "category", "eval_status"}
if not need <= set(summ.columns): sys.exit(f"evaluation_results is missing columns: {need - set(summ.columns)}")
NUM = ["accuracy", "precision", "recall", "f1", "roc_auc", "map50", "map50_95", "inference_ms", "feature_extraction_ms", "model_size_mb"]
def coerce(df):
    for c in NUM:
        if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df
summ = coerce(summ); combos = load("all_combinations.csv"); combos = coerce(combos) if combos is not None else None
yd = load("yolo_detail.csv"); imp = load("metrics/feature_importance.csv"); hist = load("metrics/dl_history.csv")
pz = RES / "metrics/predictions.npz"; P = dict(np.load(pz)) if pz.exists() else {}
ev = summ[summ.eval_status == "completed"]
notev = summ[summ.eval_status != "completed"].model.tolist()
print(f"Evaluated: {ev.model.tolist()}\nNot evaluated: {notev}")

# ---------- A. overall comparison (9 models) ----------
def bar(df, col, title, ylabel, name, sub="model_comparison", fmt="{:.3f}", x="model"):
    d = df.dropna(subset=[col])
    if d.empty: return skip(name, f"no valid '{col}' values")
    fig, ax = plt.subplots(figsize=(max(6, 0.9 * len(d) + 2), 5))
    cols = [CAT_COL.get(c, "#888") for c in d.category] if "category" in d else None
    bars = ax.bar(d[x], d[col], color=cols, edgecolor="black", linewidth=.6)
    for b, v in zip(bars, d[col]): ax.annotate(fmt.format(v), (b.get_x() + b.get_width() / 2, v), ha="center", va="bottom", fontsize=9, xytext=(0, 3), textcoords="offset points")
    ax.set_title(title); ax.set_ylabel(ylabel); ax.set_xlabel("Model"); ax.tick_params(axis="x", rotation=35)
    for t in ax.get_xticklabels(): t.set_ha("right")
    if "category" in d:
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(color=CAT_COL[c], label=c) for c in CAT_COL if c in set(d.category)], title="Category", loc="best")
    ax.set_ylim(0, max(1.0, d[col].max() * 1.1) if col not in ("inference_ms", "model_size_mb") else d[col].max() * 1.18)
    fig.text(0.01, -0.04, NOTE if col in ("accuracy", "precision", "recall", "f1", "map50", "map50_95") else "", fontsize=7, wrap=True)
    fig.tight_layout(); save(fig, sub, name)

M = [("accuracy", "Accuracy", "Accuracy"), ("precision", "Precision", "Precision"), ("recall", "Recall", "Recall"), ("f1", "F1-score", "F1-score"),
     ("map50", "mAP@0.5 (mask)", "mAP@0.5"), ("map50_95", "mAP@0.5:0.95 (mask)", "mAP@0.5:0.95"),
     ("inference_ms", "Inference time per image", "Time (ms / image)"), ("model_size_mb", "Model size", "Size (MB)")]
for i, (c, t, yl) in enumerate(M, 1):
    bar(ev, c, f"{t} comparison", yl, f"{i:02d}_{c}_comparison.png", fmt="{:.1f}" if c in ("inference_ms", "model_size_mb") else "{:.3f}")
long = ev.melt(id_vars=["model", "category"], value_vars=["accuracy", "precision", "recall", "f1"], var_name="metric", value_name="value").dropna(subset=["value"])
if len(long):
    fig, ax = plt.subplots(figsize=(max(8, 1.2 * long.model.nunique() + 3), 5.5))
    sns.barplot(data=long, x="model", y="value", hue="metric", ax=ax, edgecolor="black", linewidth=.5)
    ax.set_title("Overall model performance comparison"); ax.set_ylabel("Score"); ax.set_xlabel("Model"); ax.set_ylim(0, 1.08)
    ax.tick_params(axis="x", rotation=35); [t.set_ha("right") for t in ax.get_xticklabels()]; ax.legend(title="Metric", ncol=4, loc="lower center")
    fig.text(0.01, -0.04, NOTE, fontsize=7); fig.tight_layout(); save(fig, "model_comparison", "09_overall_performance.png")
else: skip("overall performance", "no valid values")

# ---------- B. YOLO ----------
yolo = ev[ev.category == "YOLO Segmentation"].model.tolist()
LOSS = {"train": ["train/box_loss", "train/seg_loss", "train/cls_loss", "train/dfl_loss"], "val": ["val/box_loss", "val/seg_loss", "val/cls_loss", "val/dfl_loss"]}
for key in yolo:
    rp = RES / "yolo_runs" / key / "results.csv"
    if not rp.exists(): skip(f"{key} curves", f"{rp} not found"); continue
    r = pd.read_csv(rp); r.columns = [c.strip() for c in r.columns]; e = r["epoch"]
    def line(cols, labels, title, ylabel, name):
        cols2 = [(c, l) for c, l in zip(cols, labels) if c in r.columns]
        if not cols2: return skip(f"{key} {name}", "columns missing in results.csv")
        fig, ax = plt.subplots(figsize=(7, 4.6))
        for c, l in cols2: ax.plot(e, r[c], lw=1.8, label=l)
        ax.set_title(f"{key}: {title}"); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel); ax.legend(); fig.tight_layout(); save(fig, f"yolo_segmentation/{key}", f"{name}.png")
    for split, nm in [("train", "training"), ("val", "validation")]:
        cs = [c for c in LOSS[split] if c in r.columns]
        if cs:
            r[f"{split}/total"] = r[cs].sum(axis=1)
            line(cs + [f"{split}/total"], [c.split("/")[1] for c in cs] + ["total"], f"{nm} loss", "Loss", f"{nm}_loss_curve")
    line(["train/box_loss", "val/box_loss"], ["train", "validation"], "box loss", "Loss", "box_loss_curve")
    line(["train/seg_loss", "val/seg_loss"], ["train", "validation"], "segmentation loss", "Loss", "seg_loss_curve")
    line(["metrics/mAP50(B)", "metrics/mAP50(M)"], ["box", "mask"], "mAP@0.5", "mAP", "map50_curve")
    line(["metrics/mAP50-95(B)", "metrics/mAP50-95(M)"], ["box", "mask"], "mAP@0.5:0.95", "mAP", "map50_95_curve")
    # genuine Ultralytics PR curves / confusion matrices (generated during validation)
    vd = RES / "yolo_runs" / f"{key}_val"
    for fn in ["MaskPR_curve.png", "BoxPR_curve.png", "confusion_matrix.png", "confusion_matrix_normalized.png"]:
        if (vd / fn).exists():
            dst = G / f"yolo_segmentation/{key}"; dst.mkdir(parents=True, exist_ok=True); shutil.copy(vd / fn, dst / fn); done.append(f"results/graphs/yolo_segmentation/{key}/{fn} (Ultralytics)")
        else: skip(f"{key} {fn}", "not found in validation run")
if yd is not None and len(yd):
    yd = coerce(yd.copy()); 
    for sfx, ttl in [("mask", "Segmentation (mask)"), ("box", "Detection (box)")]:
        cols = [f"{sfx}_precision", f"{sfx}_recall", f"{sfx}_f1", f"{sfx}_map50", f"{sfx}_map50_95"]
        d = yd.melt(id_vars="model", value_vars=[c for c in cols if c in yd], var_name="metric", value_name="v").dropna(subset=["v"])
        d["metric"] = d.metric.str.replace(f"{sfx}_", "").str.replace("map50_95", "mAP@0.5:0.95").str.replace("map50", "mAP@0.5").str.title().str.replace("Map", "mAP")
        if d.empty: continue
        fig, ax = plt.subplots(figsize=(8, 5)); sns.barplot(data=d, x="model", y="v", hue="metric", ax=ax, edgecolor="black", linewidth=.5)
        for c in ax.containers: ax.bar_label(c, fmt="%.3f", fontsize=7, padding=2)
        ax.set_title(f"YOLO {ttl} performance comparison"); ax.set_ylabel("Score"); ax.set_xlabel("Model"); ax.set_ylim(0, 1.1); ax.legend(ncol=3, loc="lower center")
        fig.tight_layout(); save(fig, "yolo_segmentation", f"segmentation_performance_{sfx}.png")
else: skip("YOLO performance comparison", "yolo_detail.csv missing")

# ---------- C/D. classical + deep (all combinations) ----------
def cm_panel(items, title, sub, name):
    items = [(l, k) for l, k in items if f"prob__{k}" in P]
    if not items or "y_true" not in P: return skip(name, "no saved predictions")
    n = len(items); cols = min(n, 2); rows_ = int(np.ceil(n / cols)); fig, axs = plt.subplots(rows_, cols, figsize=(4.2 * cols, 3.8 * rows_), squeeze=False)
    for ax in axs.ravel(): ax.axis("off")
    for ax, (l, k) in zip(axs.ravel(), items):
        ax.axis("on"); cm = confusion_matrix(P["y_true"], (P[f"prob__{k}"] >= .5).astype(int), labels=[0, 1])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, xticklabels=["Background", "Pothole"], yticklabels=["Background", "Pothole"])
        ax.set_title(l, fontsize=10); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    fig.suptitle(title, fontweight="bold"); fig.tight_layout(); save(fig, sub, name)

if combos is not None and len(combos):
    cl = combos[(combos.category == "Classical ML") & (combos.eval_status == "completed")]
    dl = combos[(combos.category == "Deep Learning") & (combos.eval_status == "completed")]
    if len(cl):
        MOD = [m for m in ["AdaBoost", "GradientBoosting", "XGBoost", "LightGBM"] if m in set(cl.model)]; FS = list(dict.fromkeys(cl.feature_set))
        fig, ax = plt.subplots(figsize=(max(8, 1.6 * len(FS) + 3), 5)); sns.barplot(data=cl, x="feature_set", y="accuracy", hue="model", hue_order=MOD, ax=ax, edgecolor="black", linewidth=.5)
        ax.set_title("Classical ML accuracy: all model x feature-set combinations"); ax.set_xlabel("Feature set"); ax.set_ylabel("Accuracy"); ax.set_ylim(0, 1.05); ax.tick_params(axis="x", rotation=25); ax.legend(title="Model", ncol=4, loc="lower center")
        fig.tight_layout(); save(fig, "classical_ml", "accuracy_all_combinations.png")
        for fs in FS:
            d = cl[cl.feature_set == fs].melt(id_vars="model", value_vars=["precision", "recall", "f1"], var_name="metric", value_name="v")
            fig, ax = plt.subplots(figsize=(7.5, 4.8)); sns.barplot(data=d, x="model", y="v", hue="metric", order=MOD, ax=ax, edgecolor="black", linewidth=.5)
            for c in ax.containers: ax.bar_label(c, fmt="%.3f", fontsize=7, padding=2)
            ax.set_title(f"Precision / Recall / F1 - {fs}"); ax.set_xlabel("Model"); ax.set_ylabel("Score"); ax.set_ylim(0, 1.1); ax.legend(ncol=3, loc="lower center"); fig.tight_layout(); save(fig, "classical_ml/precision_recall_f1", f"prf1_{fs}.png")
            cm_panel([(m, f"{m}__{fs}") for m in MOD], f"Confusion matrices - {fs}", "confusion_matrices", f"cm_classical_{fs}.png")
            fig, ax = plt.subplots(figsize=(5.5, 5)); ok = False
            for m in MOD:
                k = f"prob__{m}__{fs}"
                if k in P: fpr, tpr, _ = roc_curve(P["y_true"], P[k]); ax.plot(fpr, tpr, lw=1.8, label=f"{m} (AUC={auc(fpr, tpr):.3f})"); ok = True
            if ok:
                ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.set_title(f"ROC curves - {fs}"); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.legend(loc="lower right"); fig.tight_layout(); save(fig, "classical_ml/roc", f"roc_{fs}.png")
            else: plt.close(fig); skip(f"ROC {fs}", "no saved probabilities")
        for c, t in [("f1", "F1-score"), ("accuracy", "Accuracy"), ("roc_auc", "ROC-AUC")]:
            pv = cl.pivot(index="model", columns="feature_set", values=c).reindex(index=MOD, columns=FS)
            fig, ax = plt.subplots(figsize=(max(7, 1.3 * len(FS) + 2), 3.8)); sns.heatmap(pv, annot=True, fmt=".3f", cmap="YlGnBu", vmin=max(0, np.nanmin(pv.values) - .05), vmax=1, ax=ax, cbar_kws={"label": t})
            ax.set_title(f"Classification performance ({t}): model x feature set"); ax.set_xlabel("Feature set"); ax.set_ylabel("Model"); ax.tick_params(axis="x", rotation=25); fig.tight_layout(); save(fig, "classical_ml", f"performance_heatmap_{c}.png")
        fig, ax = plt.subplots(figsize=(max(7, 1.3 * len(FS) + 2), 4.6)); sns.barplot(data=cl, x="feature_set", y="f1", errorbar="sd", color="#66a182", edgecolor="black", linewidth=.5, ax=ax, capsize=.15)
        ax.set_title("Feature extraction comparison (mean F1 over the 4 classifiers, error bar = SD)"); ax.set_xlabel("Feature set"); ax.set_ylabel("F1-score"); ax.set_ylim(0, 1.05); ax.tick_params(axis="x", rotation=25); fig.tight_layout(); save(fig, "deep_learning", "feature_extraction_comparison.png")
        fe = cl.groupby("feature_set")[["feature_extraction_ms"]].first().reset_index()
        bar(fe.assign(category="Classical ML"), "feature_extraction_ms", "Feature extraction time per image", "Time (ms / image)", "feature_extraction_time.png", sub="deep_learning", fmt="{:.2f}", x="feature_set")
    else: skip("classical ML graphs", "no completed classical results")
    if imp is not None and len(imp):
        fig, ax = plt.subplots(figsize=(7.5, 4.8)); sns.barplot(data=imp, x="group", y="importance", hue="model", ax=ax, edgecolor="black", linewidth=.5)
        ax.set_title("Feature importance by feature group (Handcrafted_All, normalised)"); ax.set_xlabel("Feature group"); ax.set_ylabel("Share of total importance"); ax.legend(title="Model"); fig.tight_layout(); save(fig, "classical_ml", "feature_importance_groups.png")
    else: skip("feature importance", "feature_importance.csv missing")
    if len(dl):
        for c, t in [("accuracy", "Accuracy"), ("precision", "Precision"), ("recall", "Recall"), ("f1", "F1-score")]:
            bar(dl, c, f"MobileNetV2 vs ResNet50 - {t}", t, f"{c}_comparison.png", sub="deep_learning")
        bar(dl, "inference_ms", "MobileNetV2 vs ResNet50 - inference time", "Time (ms / image)", "inference_time_comparison.png", sub="deep_learning", fmt="{:.2f}")
        cm_panel([(m, f"{m}__end-to-end") for m in dl.model], "Confusion matrices - fine-tuned CNNs", "confusion_matrices", "cm_deep_learning.png")
    else: skip("deep learning graphs", "no completed deep-learning results")
else: skip("classical/deep graphs", "all_combinations.csv missing")
if hist is not None and len(hist):
    for mname, h in hist.groupby("model"):
        fig, axs = plt.subplots(1, 2, figsize=(10, 4.2))
        axs[0].plot(h.epoch, h.train_loss, label="train"); axs[0].plot(h.epoch, h.val_loss, label="validation"); axs[0].set_title(f"{mname}: loss"); axs[0].set_ylabel("Cross-entropy loss")
        axs[1].plot(h.epoch, h.train_acc, label="train"); axs[1].plot(h.epoch, h.val_acc, label="validation"); axs[1].set_title(f"{mname}: accuracy"); axs[1].set_ylabel("Accuracy")
        for ax in axs: ax.set_xlabel("Epoch"); ax.legend()
        fig.tight_layout(); save(fig, "deep_learning", f"training_curves_{mname}.png")
else: skip("DL training curves", "dl_history.csv missing (models not trained)")

print(f"\n=== {len(done)} graphs generated ===")
for d in done: print("  +", d)
print(f"=== {len(skipped)} skipped ===")
for s in skipped: print("  -", s)
if notev: print("Models not yet evaluated (no graphs for them):", ", ".join(notev))

In [ ]:
!python /content/pothole_project/scripts/generate_graphs.py --base /content/pothole_project

## 9. Excel workbook with all results and combinations

In [ ]:
# ===== EXCEL: every result, every combination =====
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

rd = lambda p: pd.read_csv(p) if Path(p).exists() else pd.DataFrame()
summary, combos = rd(RES / "evaluation_results.csv"), rd(RES / "all_combinations.csv")
yd, imp, hist = rd(RES / "yolo_detail.csv"), rd(RES / "metrics/feature_importance.csv"), rd(RES / "metrics/dl_history.csv")
yh = []
for d in sorted((RES / "yolo_runs").glob("*")) if (RES / "yolo_runs").exists() else []:
    if (d / "results.csv").exists():
        t = pd.read_csv(d / "results.csv"); t.columns = [c.strip() for c in t.columns]; t.insert(0, "model", d.name); yh.append(t)
yh = pd.concat(yh) if yh else pd.DataFrame()
vr = json.load(open(RES / "metrics/dataset_validation.json"))
info = pd.DataFrame([("Classes", ", ".join(vr["classes"])), ("Train images / labels", f"{vr['train']['images']} / {vr['train']['labels']}"),
                     ("Valid images / labels", f"{vr['valid']['images']} / {vr['valid']['labels']}"),
                     ("Train / valid polygon instances", f"{vr['train']['instances']} / {vr['valid']['instances']}"),
                     ("Classification patches (train)", int(len(ytr))), ("Classification patches (valid)", int(len(yva))),
                     ("Dataset validation passed", vr["passed"]), ("GPU", vr["environment"]["gpu"])], columns=["item", "value"])
notes = pd.DataFrame({"notes": [
    "Missing values are blank = not applicable / not evaluated. They are never replaced by zero.",
    "Summary_9_Models: classical models use the feature set fixed in advance (Handcrafted_All). All_Combinations holds every model x feature-set result.",
    "All metrics are measured on the validation split (no separate test split exists). No hyper-parameter or checkpoint selection used it, but picking the 'best combination' from this sheet is still optimistic.",
    "YOLO precision/recall/F1/mAP = instance MASK metrics from Ultralytics (see YOLO_Box_vs_Mask for box metrics). YOLO accuracy is not defined.",
    "MobileNetV2, ResNet50 and the boosting models are PATCH classifiers (pothole patch vs pothole-free road patch): different task from YOLO, so not strictly comparable.",
    "mAP is reported only for YOLO (valid detection/segmentation evaluation). Inference time for classical models = feature extraction + prediction (batch-1 for CNNs)."]})
order = {c: i for i, c in enumerate(["YOLO Segmentation", "Deep Learning", "Classical ML"])}
if len(combos): combos = combos.sort_values(["category", "model", "feature_set"], key=lambda s: s.map(order) if s.name == "category" else s)
piv = {}
cl = combos[combos.category == "Classical ML"] if len(combos) else combos
for c in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    if len(cl): piv[f"Classical_{c}"] = cl.pivot(index="model", columns="feature_set", values=c).reset_index()

XL = RES / "PotholeSegmentation_All_Results.xlsx"
with pd.ExcelWriter(XL, engine="openpyxl") as w:
    for name, df in [("Summary_9_Models", summary), ("All_Combinations", combos), ("YOLO_Box_vs_Mask", yd), ("Feature_Importance", imp),
                     ("DL_Training_History", hist), ("YOLO_Epoch_History", yh), *piv.items(), ("Dataset_Info", info), ("Notes", notes)]:
        if len(df): df.to_excel(w, sheet_name=name[:31], index=False)

wb = load_workbook(XL); green = PatternFill("solid", fgColor="C6EFCE")
for ws in wb:
    for c in ws[1]: c.font = Font(bold=True, color="FFFFFF"); c.fill = PatternFill("solid", fgColor="1F4E78"); c.alignment = Alignment(wrap_text=True, vertical="center")
    for i, col in enumerate(ws.columns, 1):
        ws.column_dimensions[get_column_letter(i)].width = min(60, max(10, max(len(str(c.value)) if c.value is not None else 0 for c in col) + 2))
    ws.freeze_panes = "B2"
    hdr = [c.value for c in ws[1]]
    if ws.title in ("Summary_9_Models", "All_Combinations"):          # highlight best value per metric column
        for j, h in enumerate(hdr, 1):
            if h in ("accuracy", "precision", "recall", "f1", "roc_auc", "map50", "map50_95", "inference_ms", "model_size_mb"):
                vals = [(r[j - 1].value, r[j - 1]) for r in ws.iter_rows(min_row=2) if isinstance(r[j - 1].value, (int, float))]
                if vals:
                    best = (min if h in ("inference_ms", "model_size_mb") else max)(v for v, _ in vals)
                    for v, cell in vals:
                        if v == best: cell.fill = green
                        cell.number_format = "0.000" if h not in ("inference_ms", "model_size_mb") else "0.00"
wb.save(XL)
print("Saved:", XL); display(summary)

## 10. Download

In [ ]:
# ===== package everything for download =====
shutil.make_archive("/content/pothole_results", "zip", RES)
from google.colab import files
files.download(str(RES / "PotholeSegmentation_All_Results.xlsx"))
files.download("/content/pothole_results.zip")     # all graphs (300 DPI PNG), CSV/JSON, YOLO runs, predictions